### UC Function 생성

In [0]:
%pip install unitycatalog-ai[databricks]

In [0]:
%restart_python

In [0]:
### python

# Databricks notebook or Python script
from typing import Optional
from unitycatalog.ai.core.databricks import DatabricksFunctionClient

# 카탈로그/스키마를 환경에 맞게 설정하세요
CATALOG = "sk_poc"
SCHEMA = "minjung" # <<-- 이 부분을 수정해주세요

client = DatabricksFunctionClient()  # 기본은 serverless 실행

def label_sentiment_by_rating(rating: Optional[int]) -> str:
    """
    리뷰 평점(1~5)을 한국어 감정 레이블로 변환합니다.
    - 4~5: '긍정'
    - 3: '중립'
    - 1~2: '부정'
    - 그 외/Null: '알 수 없음'

    Args:
      rating (int | None): 리뷰 평점 (1~5), Null 허용.

    Returns:
      str: 한국어 감정 레이블 문자열.
    """
    if rating is None:
        return "알 수 없음"
    if rating >= 4:
        return "긍정"
    if rating == 3:
        return "중립"
    if 1 <= rating <= 2:
        return "부정"
    return "알 수 없음"

# Unity Catalog에 함수로 등록
function_info = client.create_python_function(
    func=label_sentiment_by_rating,
    catalog=CATALOG,
    schema=SCHEMA,
    replace=True,  # 기존 함수가 있으면 교체
)

# 선택: 함수 테스트 실행
result = client.execute_function(
    function_name=f"{CATALOG}.{SCHEMA}.label_sentiment_by_rating",
    parameters={"rating": 5}
)
print(result.value)  # "긍정"

In [0]:
# 지정한 SKU의 리뷰 요약(평균 평점, 리뷰 수, 최근 리뷰 날짜, 감정 분포)을
# JSON 문자열로 반환하고, 데이터가 없으면 안내 문구를 반환합니다.

spark.sql(f"""
  CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA}.summarize_sku_reviews( -- 수정해주세요.
    in_sku_id INT COMMENT '요약할 SKU ID'
  )
  RETURNS STRING
  COMMENT '지정한 SKU에 대한 리뷰 요약(평균 평점, 리뷰 수, 최근 리뷰 날짜, 감정 분포)을 JSON 문자열로 반환합니다. 데이터가 없으면 안내 문구를 반환합니다.'
  RETURN
  SELECT COALESCE(
    (
      SELECT to_json(
        named_struct(
          'sku_id', in_sku_id,
          'review_count', COUNT(*),
          'avg_rating', ROUND(AVG(rating), 2),
          'last_review_date', MAX(review_date),
          'sentiment_counts', named_struct(
            'positive', SUM(CASE WHEN rating >= 4 THEN 1 ELSE 0 END),
            'neutral',  SUM(CASE WHEN rating = 3 THEN 1 ELSE 0 END),
            'negative', SUM(CASE WHEN rating <= 2 THEN 1 ELSE 0 END)
          )
        )
      )
      FROM {CATALOG}.{SCHEMA}.clothing_reviews
      WHERE sku_id = in_sku_id
    ),
    '해당 SKU 리뷰가 없습니다.'
  )
  """
)